# ML-04 - Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/8-BitRhyon/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane 4 (provisional): CTR and engagement opportunity scoring for content review.
This notebook writes the contract for that lane on the real warehouse release, proves three
facts about the March 2026 slice with three queries, builds a five-feature page-month frame,
and then springs the label-leak trap on purpose.

- Warehouse: `FlyRank/internship-warehouse`, build `v20260703`
- Development month: 2026-03 (features) into 2026-04 (label)
- Final month 2026-06 stays sealed


## 1. Unit of analysis + time window

**One row = one content page, for one client, in one calendar month (a page-month).**

At the warehouse's daily grain, one row is `report_date x client_hash_id x content_hash_id`.
My decision frame rolls those daily rows up to one page-month, because the decision I support
is made once per page per month: which pages should a reviewer look at first next month.

| # | Contract question | Answer |
|---|---|---|
| 1 | What one row means | One `content_hash_id` for one `client_hash_id`, aggregated over one calendar month. |
| 2 | Tables | `fact_content_daily_performance` for both months, plus `dim_content` for page age. `dim_clients` is context for coverage only. |
| 3 | Time window | Features: 2026-03-01 to 2026-03-31. Label: 2026-04-01 to 2026-04-30. Decision moment: 2026-04-01. 2026-06 stays sealed. |
| 4 | Label or proxy | `low_ctr_next` = 1 when April CTR is below the March CTR median of the page's own March position tier, for pages with at least 100 April impressions. Thresholds are fixed on April 1 from March data only. |
| 5 | One deliberate exclusion | GA4 engagement: `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `scroll_events`. Q3 shows `ga4_data_available IS TRUE` on only 4.2% of March rows, and coverage is patchy inside a page. Requiring it would shrink and bias the slice. |

The label is a ranking proxy, not a revenue number. It says "this page is likely to turn
attention into clicks worse than its position peers next month", which is what a review queue needs.


In [1]:
try:
    import duckdb
except ImportError:  # Colab images do not always ship duckdb
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

import os
import numpy as np
import pandas as pd

# Token order: env var -> Colab Secret -> local hf login. Never print it, never paste it in a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from huggingface_hub import get_token
        HF_TOKEN = get_token()
    except Exception:
        HF_TOKEN = None

assert HF_TOKEN, "No Hugging Face token. Set a Colab Secret named HF_TOKEN, or run `hf auth login` once."

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

BASE = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
APR = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/data_0.parquet')"

print("connected to", BASE)
print("token loaded: yes (value never printed)")


connected to hf://datasets/FlyRank/internship-warehouse
token loaded: yes (value never printed)


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `gsc_sum_position`, `report_date`, `content_created_date` | All measured during March 2026, before the April 1 decision moment. |
| **Label / proxy** | `low_ctr_next`, built from April 2026 `gsc_impressions` and `gsc_clicks` | The outcome month. It is never a feature. |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `month`, `tier` | Used for grouping, joining, and the client-grouped split. Never learned from. |
| **Excluded** | GA4 engagement columns, the query table, client identity as a feature | Each one gets a why below. |

Excluded, with the why:

- **GA4 engagement** (`ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `scroll_events`): Q3 below shows how few page-days carry it. A model that needs it would drop most rows and keep the clients with the cleanest analytics, which is a selection effect, not a signal.
- **Query table** (`fact_content_query_90d`): its 90-day window overlaps March and April, and its per-content columns repeat on every query row, so a careless `SUM()` inflates them. It stays out of the Week 3 feature set.
- **Client identity**: a hash is a group key, not evidence. Using it as a feature would let the model memorise clients, which the client-grouped split is designed to catch.

The next cell encodes this contract as code, so a later edit cannot quietly add a sixth feature
or promote the label source into the feature list.


In [2]:
FEATURES = ["ctr_pct", "avg_pos", "march_impressions_log", "position_sd", "content_age_days"]
LABEL = "low_ctr_next"
CONTEXT = ["client_hash_id", "content_hash_id", "tier"]
LEAK_SOURCE = "april_ctr_pct"

assert len(FEATURES) == 5, "the card allows five features at most"
assert LEAK_SOURCE not in FEATURES, "the label source must never be a feature"

print("features:", FEATURES)
print("label:", LABEL)
print("leak source held out of the feature list:", LEAK_SOURCE)


features: ['ctr_pct', 'avg_pos', 'march_impressions_log', 'position_sd', 'content_age_days']
label: low_ctr_next
leak source held out of the feature list: april_ctr_pct


## 3. Verify it with queries

Three claims, three queries, all on the March 2026 partition. The queries only describe the
feature month, so nothing here touches April.

- **Q1, grain**: one row really is `report_date x client_hash_id x content_hash_id`.
- **Q2, slice**: how many page-days my page-months are built from, and their date span.
- **Q3, availability**: what survives an `IS TRUE` filter on the two availability flags.

The slice floor is deliberate: a page needs at least 500 March impressions and at least 20 March
days with GSC data before a CTR ratio is worth reading. That is a decision-support choice, not a
statistical law.


In [3]:
q1 = con.execute(f"""
    SELECT
        COUNT(*)                                                       AS rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys,
        COUNT(DISTINCT report_date)                                    AS days
    FROM {MAR}
""").df()
q1


,rows,distinct_keys,days
0,9841378,9841378,31


**Q1 result.** Rows equal distinct keys, so the daily grain holds and nothing is double counted
before the roll-up. Rolling up with `GROUP BY (client_hash_id, content_hash_id)` therefore gives
exactly one row per page-month by construction.


In [4]:
q2 = con.execute(f"""
    WITH page_month AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp,
               COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_gsc
        FROM {MAR}
        GROUP BY 1, 2
    ), eligible AS (
        SELECT * FROM page_month
        WHERE imp >= 500 AND days_gsc >= 20
    )
    SELECT
        (SELECT COUNT(*) FROM eligible)                               AS page_months,
        COUNT(*)                                                      AS page_days,
        COUNT(DISTINCT d.client_hash_id || '|' || d.content_hash_id)  AS pages,
        COUNT(DISTINCT d.client_hash_id)                              AS clients,
        MIN(d.report_date)                                            AS first_day,
        MAX(d.report_date)                                            AS last_day
    FROM {MAR} d
    JOIN eligible e USING (client_hash_id, content_hash_id)
    WHERE d.gsc_data_available IS TRUE
""").df()
q2


,page_months,page_days,pages,clients,first_day,last_day
0,59306,1790518,59306,32,2026-03-01,2026-03-31


**Q2 result.** The slice is one month wide, from 2026-03-01 to 2026-03-31, and it is built from
the page-days of the eligible pages only. The page count and the page-month count match, which is
the expected shape when one month is aggregated.


In [5]:
q3 = con.execute(f"""
    SELECT 1 AS ord, 'no filter' AS availability_check, COUNT(*) AS rows
    FROM {MAR}
    UNION ALL
    SELECT 2, 'gsc_data_available IS TRUE', COUNT(*)
    FROM {MAR} WHERE gsc_data_available IS TRUE
    UNION ALL
    SELECT 3, 'ga4_data_available IS TRUE', COUNT(*)
    FROM {MAR} WHERE ga4_data_available IS TRUE
    UNION ALL
    SELECT 4, 'both IS TRUE', COUNT(*)
    FROM {MAR} WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
    ORDER BY ord
""").df()
q3["share_of_rows_pct"] = (100 * q3["rows"] / q3.loc[0, "rows"]).round(2)
q3.drop(columns="ord")


,availability_check,rows,share_of_rows_pct
0,no filter,9841378,100.00
1,gsc_data_available IS TRUE,3611061,36.69
2,ga4_data_available IS TRUE,413966,4.21
3,both IS TRUE,364347,3.70


**Q3 result.** Availability is not a formality. Most March rows do not carry GSC data, and GA4
data sits on a small minority of rows. That is the measured reason the engagement half of Lane 4
is excluded from this Week 3 contract: keeping it would cut the slice to the clients with the
cleanest analytics rather than to the pages that matter.


### 3.1 Five features, and when each one is knowable

| Feature | What it is | Knowable at the decision moment because... |
|---|---|---|
| `ctr_pct` | March clicks / March impressions x 100 | Both counters are written by GSC daily during March, so the full-month ratio exists before April 1. |
| `avg_pos` | Impression-weighted mean of `gsc_avg_position` over March days that report a position | Every daily position value is recorded in March, and the weights are March impressions. |
| `march_impressions_log` | `log1p` of March impressions | March impressions are a closed total on March 31. |
| `position_sd` | Standard deviation of daily `gsc_avg_position` during March | It is computed only from March days, so it carries no April information. |
| `content_age_days` | Days from `dim_content.content_created_date` to 2026-03-31 | The creation date is fixed before March and cannot change afterwards. |

Five, not more. Everything that needs April (clicks, impressions, CTR, position) is label-side.


In [6]:
def month_rollup(month):
    """One row per page-month, from one daily fact partition."""
    return f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp,
               SUM(gsc_clicks) AS clk,
               SUM(gsc_sum_position) AS pos_sum,
               SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions END) AS pos_imp,
               COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_gsc,
               STDDEV_SAMP(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_sd
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={month}/data_0.parquet')
        GROUP BY 1, 2
    )
    SELECT client_hash_id, content_hash_id, imp, clk, days_gsc, pos_sd,
           pos_sum / NULLIF(pos_imp, 0) AS avg_pos,
           100.0 * clk / NULLIF(imp, 0) AS ctr_pct
    FROM daily
    """

march = con.execute(month_rollup("2026-03")).df()
april = con.execute(month_rollup("2026-04")).df()

frame = march[(march.imp >= 500) & (march.days_gsc >= 20) & march.avg_pos.notna()].copy()
frame["tier"] = pd.cut(frame.avg_pos, [0, 3, 10, 20, np.inf],
                       labels=["top_3", "page_1", "page_2", "deep"]).astype(str)

# Label rule, fixed on April 1 from March data only.
thresholds = frame.groupby("tier").ctr_pct.median()

april_small = april.rename(columns={"imp": "apr_imp", "ctr_pct": "april_ctr_pct"})[
    ["client_hash_id", "content_hash_id", "apr_imp", "april_ctr_pct"]]
frame = frame.merge(april_small, on=["client_hash_id", "content_hash_id"], how="inner")
frame = frame[(frame.apr_imp >= 100) & frame.april_ctr_pct.notna()].copy()

frame["low_ctr_next"] = (frame.april_ctr_pct < frame.tier.map(thresholds)).astype(int)

dim = con.execute(
    f"SELECT client_hash_id, content_hash_id, content_created_date "
    f"FROM read_parquet('{BASE}/dim_content.parquet')"
).df()
frame = frame.merge(dim, on=["client_hash_id", "content_hash_id"], how="left")

frame["march_impressions_log"] = np.log1p(frame.imp)
frame["position_sd"] = frame.pos_sd.fillna(0)
frame["content_age_days"] = (
    pd.Timestamp("2026-03-31") - pd.to_datetime(frame.content_created_date)
).dt.days
frame["content_age_days"] = frame.content_age_days.fillna(frame.content_age_days.median())

print(f"page-months in frame: {len(frame):,}")
print(f"clients in frame: {frame.client_hash_id.nunique()}")
print(f"label base rate: {frame[LABEL].mean():.3f}")
print("\nMarch CTR median per tier (the April-1 thresholds):")
print(thresholds.round(4).to_string())
frame[["tier"] + FEATURES + [LABEL]].head(5)


page-months in frame: 58,477
clients in frame: 31
label base rate: 0.572

March CTR median per tier (the April-1 thresholds):
tier
deep      0.0741
page_1    0.2109
page_2    0.1642
top_3     0.2322


,tier,ctr_pct,avg_pos,march_impressions_log,position_sd,content_age_days,low_ctr_next
0,page_1,0.151409,5.317737,10.206551,1.022831,195,1
1,page_1,0.000000,5.205342,6.436150,3.675730,195,0
2,page_1,0.542169,4.767470,7.415175,1.523518,195,0
3,page_2,0.047416,15.327643,7.654443,6.843598,195,1
4,top_3,0.535396,2.274242,8.120589,0.851117,195,1


### 3.2 The trap: one label-derived column

The label is built from April CTR. If April CTR also sits in the feature list, the model is handed
the answer. This is the same trap as notebook 02, on warehouse data, and it is worth seeing once
because the score looks excellent and means nothing.

The next two cells add the column, score it, then delete it and keep the honest number.


In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

X = frame[FEATURES].fillna(0)
y = frame[LABEL]
groups = frame["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

assert not set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]), "client leak between train and test"


def auc_score(cols):
    model = HistGradientBoostingClassifier(max_iter=200, random_state=42)
    model.fit(frame.iloc[train_idx][cols], y.iloc[train_idx])
    proba = model.predict_proba(frame.iloc[test_idx][cols])[:, 1]
    return roc_auc_score(y.iloc[test_idx], proba)


honest_auc = auc_score(FEATURES)
print(f"train pages {len(train_idx):,} / clients {groups.iloc[train_idx].nunique()}")
print(f"test  pages {len(test_idx):,} / clients {groups.iloc[test_idx].nunique()}")
print(f"HONEST AUC (5 features, unseen clients): {honest_auc:.4f}")


train pages 27,231 / clients 21
test  pages 31,246 / clients 10
HONEST AUC (5 features, unseen clients): 0.8294


**Honest score, read plainly.** AUC is a ranking score: 0.5 is a coin flip, 1.0 is a perfect
ordering. The number below is what the queue is allowed to claim, measured on clients the model
never saw in training.


In [8]:
leaky_features = FEATURES + ["april_ctr_pct"]
leaky_auc = auc_score(leaky_features)

print(f"AUC with the label source added: {leaky_auc:.4f}")
print(f"jump over the honest score: {leaky_auc - honest_auc:+.4f}")
print("this is the trap: the column is computed from the same April clicks the label uses")


AUC with the label source added: 1.0000
jump over the honest score: +0.1706
this is the trap: the column is computed from the same April clicks the label uses


In [9]:
DELETED = "april_ctr_pct"
assert DELETED in frame.columns
frame = frame.drop(columns=[DELETED])
assert DELETED not in frame.columns, "the leak column must be gone"

restored_auc = auc_score(FEATURES)

print(f"columns remaining: {len(frame.columns)}")
print(f"AUC after deleting {DELETED}: {restored_auc:.4f}")
print(f"honest number kept:             {honest_auc:.4f}")
assert abs(restored_auc - honest_auc) < 1e-9

# The label itself stays, because it is the thing being predicted. Only its source column is removed.
assert LABEL in frame.columns
print("label column present, its source column removed: correct")


columns remaining: 15
AUC after deleting april_ctr_pct: 0.8294
honest number kept:             0.8294
label column present, its source column removed: correct


**What the trap teaches.** The score jumped toward perfect because the feature and the label were
built from the same April clicks. No model that needs the outcome month can run on April 1. After
deleting the column the honest score returns, and that is the number this lane is allowed to quote.

The rule for the coming weeks: if an AUC looks too good, check the feature list for anything
computed from the outcome window before checking the model.


## 4. Data limits

**One named limitation: this slice only speaks for pages that already get search attention.**
A page needs 500 March impressions and 20 March days of GSC data to enter the frame, and 100 April
impressions to receive a label. Everything below that floor, which is most of the catalogue, is
outside the contract.

Other limits, measured rather than guessed:

- **Client coverage is partial.** The slice uses a few dozen clients with usable GSC data out of the 104 clients in `dim_clients`. Some clients have no search or analytics access at all. The next cell counts both.
- **A client-grouped split is honest but small.** The test set holds a handful of clients, so one unusual client can move the score. That is a reason to report the number directionally, not as a production estimate.
- **Position is an average, not a rank.** `gsc_avg_position` is impression-weighted, so two pages at "position 6" can have very different day-to-day distributions. `position_sd` exists to expose part of that.
- **One forward month.** April 2026 is a single month and sits right before the sealed June month. The honest AUC is a one-month forward estimate, not proof of stability.
- **CTR is search-only.** It says nothing about on-page engagement, which is why the engagement half of Lane 4 waits for better GA4 coverage.


In [10]:
limits = con.execute(f"""
    WITH page_month AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp,
               COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_gsc
        FROM {MAR}
        GROUP BY 1, 2
    ), eligible AS (
        SELECT * FROM page_month WHERE imp >= 500 AND days_gsc >= 20
    )
    SELECT
        (SELECT COUNT(*) FROM page_month)                                     AS page_months_all,
        (SELECT COUNT(*) FROM eligible)                                       AS page_months_in_slice,
        (SELECT ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM page_month), 1)
         FROM eligible)                                                       AS slice_share_pct,
        (SELECT COUNT(DISTINCT client_hash_id) FROM eligible)                 AS clients_in_slice,
        (SELECT COUNT(*) FROM read_parquet('{BASE}/dim_clients.parquet'))     AS clients_in_dim,
        (SELECT COUNT(*) FROM read_parquet('{BASE}/dim_clients.parquet')
         WHERE access_profile = 'no_search_or_analytics_access')              AS clients_without_access
""").df()
limits


,page_months_all,page_months_in_slice,slice_share_pct,clients_in_slice,clients_in_dim,clients_without_access
0,331437,59306,17.9,32,104,26


**Reading the limit cell.** The share column is the size of the contract in one number. The
remaining pages are not bad data; they are simply too small for a monthly CTR ratio to mean much,
and this lane makes no claim about them.

The client columns show the same limit from the other side. The warehouse has more clients than the
slice can use, because history depth and access differ per client, and a group of clients has no
search or analytics access at all.

This cell measures a limitation. It is not one of the three verification queries.


## Self-check

- [x] Every section above is filled: markdown reasoning and the code that backs it
- [x] The notebook ran top to bottom with no errors (executed before commit)
- [x] Exactly three verification queries, all on the March 2026 partition, availability checked with `IS TRUE`
- [x] Five features at most, each with a "knowable at the decision moment because..." line
- [x] The label-derived column was added, scored, deleted, and the honest number kept
- [x] One named limitation of the slice, with the share measured in the notebook
- [x] No client names, URLs, private queries, or tokens anywhere in the notebook or its output
- [x] June 2026 was never used for development; it remains the sealed final month
- [x] Careful words throughout: observed, measured, directional, decision-support
